# Image Preprocessing

In [ ]:
import cv2
import numpy as np
from scipy.signal import convolve2d

def custom_preprocess_image(image_path):
    # Load the image in grayscale with error handling
    try:
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        #print(f"Loaded image shape: {image.shape}, min pixel value: {np.min(image)}, max pixel value: {np.max(image)}")

        # Check if the image was loaded successfully
        if image is None:
            raise Exception("Error loading the image.")

        # Ensure the input image is 8-bit (CV_8UC1)
        if image.dtype != np.uint8:
            image = np.uint8(image)

        # Apply imadjust (Contrast Stretching)
        min_val = np.min(image)
        max_val = np.max(image)
        adjusted_image = (image - min_val) / (max_val - min_val) * 255.0

        # Apply Weiner filter
        psf = np.ones((5, 5)) / 25  # Define a simple PSF (customizable)
        #print(f"Before Weiner filter: min={np.min(adjusted_image)}, max={np.max(adjusted_image)}")
        filtered_image = convolve2d(adjusted_image, psf, 'same')
        #print(f"After Weiner filter: min={np.min(filtered_image)}, max={np.max(filtered_image)}")

        # Ensure the filtered image is 8-bit (CV_8UC1)
        if filtered_image.dtype != np.uint8:
            filtered_image = np.uint8(filtered_image)

        # Apply histogram equalization using CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        equalized_image = clahe.apply(filtered_image)

        # Resize the image to the desired dimensions (e.g., 224x224)
        image_size = (224, 224)
        resized_image = cv2.resize(equalized_image, image_size)

        if image is None:
            raise Exception("Error during preprocessing: Image became None.")

        # Expand the single-channel image to three channels (RGB format)
        rgb_image = cv2.cvtColor(resized_image, cv2.COLOR_GRAY2RGB)

        # Normalize pixel values to [0, 1]
        normalized_image = rgb_image / 255.0

        return normalized_image

    except Exception as e:
        print(f"Error preprocessing the image: {str(e)}")
        return None

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import matplotlib.pyplot as plt

# Define paths
dataset_path = "Dataset_2_sample"
output_path = "Processed_Dataset_2_sample"  # Change this to your desired output directory

# Create the output directory if it doesn't exist
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Define subdirectories in the dataset
subdirectories = ["train", "test"]

# Loop through subdirectories
for subdir in subdirectories:
    input_subdir = os.path.join(dataset_path, subdir)
    output_subdir = os.path.join(output_path, subdir)

    # Create the output subdirectory if it doesn't exist
    if not os.path.exists(output_subdir):
        os.makedirs(output_subdir)

    # Define classes (NORMAL and PNEUMONIA)
    classes = ["NORMAL", "PNEUMONIA"]

    # Loop through classes
    for class_name in classes:
        input_class_path = os.path.join(input_subdir, class_name)
        output_class_path = os.path.join(output_subdir, class_name)

        # Create the output class subdirectory if it doesn't exist
        if not os.path.exists(output_class_path):
            os.makedirs(output_class_path)

        # Process each image in the class directory
        for filename in os.listdir(input_class_path):
            input_image_path = os.path.join(input_class_path, filename)
            output_image_path = os.path.join(output_class_path, filename)


            original_image = cv2.imread(input_image_path, cv2.IMREAD_GRAYSCALE)
            plt.subplot(1, 2, 1)
            plt.imshow(original_image, cmap='gray')
            plt.title("Original Image")
            plt.axis('off')
            # Preprocess the image
            preprocessed_image = custom_preprocess_image(input_image_path)

            plt.subplot(1, 2, 2)
            plt.imshow(preprocessed_image, cmap='gray')
            plt.title("Preprocessed Image")
            plt.axis('off')
            plt.show()
            # Save the preprocessed image

            # Scale the pixel values back to [0, 255]
            scaled_preprocessed_image = (preprocessed_image * 255).astype(np.uint8)

            #print(f"Preprocessed image shape: {preprocessed_image.shape}, min pixel value: {np.min(preprocessed_image)}, max pixel value: {np.max(preprocessed_image)}")
            if preprocessed_image is not None:
                cv2.imwrite(output_image_path, scaled_preprocessed_image)

print("Image preprocessing and saving completed.")


FileNotFoundError: [Errno 2] No such file or directory: 'Dataset_2_sample/train/NORMAL'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')